# **Differential Expression Analysis**

This notebook performs differential gene expression analysis to identify the genes that vary the most between samples with TP53 mutations versus wild-type TP53. The analysis uses the PyDESeq2 package.

In RNA-Seq data analysis, we obtain raw read counts for each gene across samples. However, raw counts are not directly comparable across samples, because of differences in sequencing depth, composition bias, and other factors. Additionally, biological variability and technical noise must be accounted for. Thus, DESeq2 provides a statistical framework to identify differentially expressed genes (DEGs) across conditions (e.g., mutated vs wild-type), while accounting for these biases.

## **Imports and Setup**

In [9]:
SCIPY_ARRAY_API=1
import pandas as pd

In [10]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PYTHONWARNINGS="ignore"


## **Data Loading and Preprocessing**

In [11]:
mutation = pd.read_csv('OmicsSomaticMutations.csv')
expression = pd.read_csv('OmicsExpressionProteinCodingGenesTPMLogp1.csv')

expression = expression.rename(columns={'Unnamed: 0': 'ModelID'})
mutation = mutation[mutation.HugoSymbol == 'TP53'] # filter for TP53 mutations
expression['Mutation'] = expression['ModelID'].isin(mutation['ModelID']).astype(int) 

expression.drop(columns=['ModelID'], inplace=True)
expression['Mutation'] = expression['Mutation'].map({0: "WT", 1: "MUT"}).astype("category")

This section loads two datasets:
- Somatic mutations data - filtered to only TP53 mutations
- Gene expression data in TPM (Transcripts Per Million) log(p+1) format

A new column 'Mutation' is added to the expression dataframe to indicate which samples have a TP53 mutation (1) and which don't (0).

We convert the numeric mutation indicator to categorical values: "WT" (wild-type) for non-mutated samples and "MUT" for samples with TP53 mutations

In [12]:
samples_to_keep = ~expression.Mutation.isna()
expression = expression.loc[samples_to_keep]

## **Single Factor Analysis**
In this analysis, we use the mutation column as our design factor. That is, we compare gene expressions of samples that have mutation to those that don't. We start by creating a DeseqDataSet object from the data with:
- counts: gene expression data (integer counts)
- metadata: mutation status information
- design: formula indicating we want to find expression differences based on mutation status
- refit_cooks: option to identify outliers
- inference: configuration for parallel processing.

In [14]:
data = expression.drop(columns=['Mutation'])
raw_data = 2 ** data - 1

In [15]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=raw_data.astype(int),
    metadata=expression[['Mutation']],
    design="~Mutation",
    refit_cooks=True,
    inference=inference,
)

Now we can run the DESeq2 algorithm which will:
1. Estimate size factors (Normalization): 
    - corrects for differences in library size and sequencing depth.
    - computes a scaling factor per sample, making counts comparable across samples.
    - uses a median-of-ratios method to account for composition bias.
2. Estimate dispersion (Variance modeling):
    - in RNA-Seq data, variability between replicates depends on the mean count level, following a negative binomial (NB) distribution.
    - dispersion reflects the extra-Poisson variability (i.e., biological variability).
    - DESeq2 estimates the dispersion per gene and shrinks it toward a global trend to stabilize estimates, especially for low-count genes.
3. Fit negative binomial generalized linear model (GLM):
    - DESeq2 fits a NB GLM for each gene.
    - Models how gene counts depend on your experimental design (e.g., condition, batch).
    - Computes log2 fold changes, p-values, and adjusted p-values (FDR corrected).

In [16]:
dds.deseq2()


Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.89 seconds.

Fitting dispersions...
... done in 9.81 seconds.

Fitting dispersion trend curve...
... done in 0.53 seconds.

Fitting MAP dispersions...
... done in 12.68 seconds.

Fitting LFCs...
... done in 6.51 seconds.

Calculating cook's distance...
... done in 3.55 seconds.

Replacing 3259 outlier genes.

Fitting dispersions...
... done in 1.80 seconds.

Fitting MAP dispersions...
... done in 1.77 seconds.

Fitting LFCs...
... done in 1.75 seconds.



Parameters are stored according to the AnnData data structure, with key-based data fields. In particular,
- X stores the count data,
- obs stores design factors,
- obsm stores sample-level data, such as "design_matrix" and "size_factors",
- varm stores gene-level data, such as "dispersions" and "LFC".

Here is how we would access dispersions and LFCs (in natural log scale):


In [17]:
print(dds.varm["dispersions"])

[1.10086469e+00 1.63890130e+02 1.76430263e-01 ...            nan
 1.67300000e+03 7.22112777e+02]


The line above displays the dispersion estimates for each gene. Dispersions represent the variability of gene counts beyond what would be expected from Poisson sampling.

In [18]:
print(dds.varm["LFC"])

                    Intercept  Mutation[T.WT]
TSPAN6 (7105)        2.794503       -0.235604
TNMD (64102)        -1.570407        0.227213
DPM1 (8813)          4.620350       -0.083864
SCYL3 (57147)        1.338852        0.050138
C1orf112 (55732)     2.529794       -0.064671
...                       ...             ...
ELOA3B (728929)           NaN             NaN
NPBWR1 (2831)       -1.033439       -0.333045
ELOA3D (100506888)        NaN             NaN
ELOA3 (162699)      -1.714421        0.045171
CDR1 (1038)         -1.702252        0.056292

[19193 rows x 2 columns]


The cell above shows the log-fold changes (LFC) for each gene. The "Intercept" column represents the baseline expression level, and "Mutation[T.WT]" shows the effect of wild-type TP53 compared to mutated.

## **Statistical Testing**

Now that dispersions and LFCs were fitted, we may proceed with statistical tests to compute p-values and adjusted p-values for differential expresion. This is the role of the DeseqStats class. It has two mandatory arguments:
- dds, which should be a fitted DeseqDataSet object,
- contrast, which is a list of three strings of the form ["variable", "tested_level", "control_level"], or directly a contrast vector.

In [19]:
ds = DeseqStats(dds, contrast=["Mutation", "MUT", "WT"], inference=inference)

PyDESeq2 computes p-values using Wald tests. This can be done using the summary() method, which runs the whole statistical analysis, cooks filtering and multiple testing adjustement included.

In [20]:
ds.summary()

Running Wald tests...


Log2 fold change & Wald test p-value: Mutation MUT vs WT
                     baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6 (7105)       14.960051        0.339905  0.077685  4.375408  0.000012   
TNMD (64102)         0.113498       -0.327799  0.930039 -0.352457  0.724495   
DPM1 (8813)         98.293105        0.120991  0.031015  3.900982  0.000096   
SCYL3 (57147)        3.875504       -0.072333  0.039115 -1.849273  0.064418   
C1orf112 (55732)    12.070711        0.093300  0.036674  2.544054  0.010957   
...                       ...             ...       ...       ...       ...   
ELOA3B (728929)      0.000000             NaN       NaN       NaN       NaN   
NPBWR1 (2831)        0.255374        0.480482  0.360305  1.333542  0.182354   
ELOA3D (100506888)   0.000000             NaN       NaN       NaN       NaN   
ELOA3 (162699)       0.000511       -0.065168  2.937512 -0.022185  0.982301   
CDR1 (1038)          0.009183       -0.081212  1.934009 -0.041992  0.96650

... done in 1.63 seconds.



The summary table has different columns which represent:
- baseMean: average expression level across all samples
- log2FoldChange: log2 ratio of expression in MUT vs WT (positive = higher in MUT, negative = higher in WT)
- lfcSE: standard error of the log2 fold change estimate
- stat: Wald test statistic
- pvalue: raw p-value
- padj: adjusted p-value (corrected for multiple testing)

For example, the gene TSPAN6 is significantly upregulated in TP53 MUT samples:
- log2FC = 0.34 $\to$ ~12% increase (2^0.34 ≈ 1.26)
- p-value < 0.05 $\to$ considered statistically significant


In [21]:
degs = ds.results_df[ds.results_df['padj'] < 0.05]
degs = degs.sort_values(by = 'log2FoldChange', ascending = False)

degs.to_csv('DEGs_filtered_sorted.csv', index=True)


In [22]:
top_genes = degs.sort_values(by='log2FoldChange', ascending=False)

In [23]:
top_genes.sort_values(by='padj', ascending=True, inplace=True)

In [24]:
results = ds.results_df

| Column Name      | Meaning                                                               |
| ---------------- | --------------------------------------------------------------------- |
| `baseMean`       | Mean of normalized counts for the gene across all samples.            |
| `log2FoldChange` | Estimated **log₂ fold change** (MUT vs WT). Positive = higher in MUT. |
| `lfcSE`          | Standard error of the log₂ fold change estimate.                      |
| `stat`           | Wald test statistic for differential expression.                      |
| `pvalue`         | Raw p-value from the Wald test.                                       |
| `padj`           | Adjusted p-value (FDR, Benjamini-Hochberg corrected).                 |


In [25]:
results.sort_values(by ='padj', ascending=True, inplace=True)

In [26]:
results

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
MDM2 (4193),68.501599,-2.198735,0.070996,-30.970005,1.366703e-210,2.154471e-206
PMEL (6490),127.353123,-5.113751,0.185439,-27.576505,2.129363e-167,1.678364e-163
ZMAT3 (64393),8.258558,-1.609287,0.059845,-26.890775,2.815665e-159,1.479538e-155
RPS27L (51065),131.129726,-1.206544,0.049139,-24.553740,3.944908e-133,1.554688e-129
CDKN1A (1026),87.447080,-1.777482,0.076288,-23.299628,4.471865e-120,1.409890e-116
...,...,...,...,...,...,...
C8orf44-SGK3 (100533105),0.089396,-0.022747,0.389179,-0.058448,9.533920e-01,NaN
ELOA3B (728929),0.000000,NaN,NaN,NaN,NaN,NaN
ELOA3D (100506888),0.000000,NaN,NaN,NaN,NaN,NaN
ELOA3 (162699),0.000511,-0.065168,2.937512,-0.022185,9.823006e-01,NaN


In [27]:
pos_corr = degs[:10]

neg_corr = degs[-10:]

In [28]:
pos_corr

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
SCGB1A1 (7356),3.883678,4.280461,0.558748,7.660815,1.847574e-14,3.111661e-13
KLK5 (25818),18.685852,4.085804,0.382011,10.695521,1.068179e-26,1.122585e-24
LGALS4 (3960),24.484483,3.891671,0.387099,10.053425,8.872928e-24,6.789943e-22
GRP (2922),3.226136,3.598727,0.829901,4.336335,1.448783e-05,5.157773e-05
KRT5 (3852),144.877821,3.517110,0.319169,11.019598,3.074273e-28,3.815971e-26
REG1A (5967),1.867809,3.459137,0.653676,5.291818,1.211062e-07,6.353137e-07
HEPACAM2 (253012),2.652083,3.360506,0.562645,5.972690,2.333727e-09,1.637244e-08
KRT13 (3860),33.819558,3.352560,0.374957,8.941196,3.849818e-19,1.441533e-17
SCGN (10590),1.969392,3.175753,0.580816,5.467747,4.557916e-08,2.572538e-07
ASCL1 (429),7.687036,3.156794,0.685904,4.602384,4.176815e-06,1.651974e-05


**SCGB1A1** (Secretoglobin Family 1A Member 1): codifies for secretoglobin (which is a anti infiammatory protein); considered a biomarker for lung diseases.

**LGALS4** (Galectin-4): involved immune response modulation.

**GRP** (Gastrin-Releasing Peptide): mitogenic factor (i.e. stimulates cells to do mitosis) in various cancers.

**KRT5** (Keratin 5): marker for basal-like breast cancer, often associated with TP53 mutations. (!!)

**REG1A** (Regenerating Family Member 1 Alpha): overexpressed in breast cancer; enhances chemo- and radiosensitivity in certain cancers.

**KRT13** (Keratin 13): promotes breast cancer cell growth and metastasis through the plakoglobin/c-Myc pathway.

**ASCL1** (Achaete-Scute Family BHLH Transcription Factor 1): transcription factor involved in neuroendocrine differentiation. It is known that elevated ASCL1 expression correlates with higher TP53 mutation rates in breast cancer.

In [ ]:
neg_corr

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
SLC45A2 (51151),5.492469,-3.581781,0.577926,-6.197646,5.731386e-10,4.508461e-09
WFDC1 (58189),3.791994,-3.652714,0.418721,-8.723512,2.697042e-18,8.641500e-17
PODN (127435),1.746737,-3.670238,0.411117,-8.927489,4.357887e-19,1.616417e-17
COX7A1 (1346),11.446363,-3.811108,0.539986,-7.057797,1.691627e-12,2.081304e-11
TRPM1 (4308),3.179003,-3.847186,0.729146,-5.276292,1.318242e-07,6.869673e-07
S100B (6285),36.782323,-3.871791,0.442792,-8.744042,2.249159e-18,7.310461e-17
ACAN (176),2.662884,-4.067066,0.534313,-7.611770,2.703669e-14,4.467572e-13
TYR (7299),15.440760,-4.137726,0.824351,-5.019371,5.184100e-07,2.414244e-06
MLANA (2315),18.985175,-4.727944,0.716646,-6.597319,4.186591e-11,3.926081e-10
PMEL (6490),127.353123,-5.113751,0.185439,-27.576505,2.129363e-167,1.678364e-163


**WFDC1** (WAP Four-Disulfide Core Domain 1): functions as a tumor suppressor; downregulated in various cancers.

**COX7A1** (Cytochrome c Oxidase Subunit 7A1): component of the mitochondrial respiratory chain; its expression is reduced in certain cancers.

**TRPM1** (Transient Receptor Potential Melastatin 1): functions as a tumor suppressor: